In [1]:
import pandas as pd
import numpy as np

In [2]:
sg = pd.read_csv(
    "../data/raw/wvs7_singapore.csv",
    sep=";",
    index_col=False
)

sg.head()


,version,doi,A_YEAR,B_COUNTRY,B_COUNTRY_ALPHA,C_COW_NUM,C_COW_ALPHA,D_INTERVIEW,FW_START,FW_END,...,WEIGHT4A,WEIGHT4B,RESEMAVALBWGT,RESEMAVALWGT,SECVALBWGT,Y001_1,Y001_2,Y001_3,Y001_4,Y001_5
0,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070001.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,0.83,0.0,0.0,0.0,1.0,0.0
1,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070002.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,0.83,1.0,1.0,1.0,0.0,1.0
2,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070003.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,1.00,1.0,0.0,1.0,0.0,1.0
3,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070004.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,1.00,1.0,0.0,0.0,0.0,0.0
4,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070005.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,1.00,0.0,0.0,0.0,0.0,0.0


In [4]:
variables_to_check = [
    "Q131",
    "Q138",
    "Q260",
    "Q262"
]

sg[variables_to_check].head()

,Q131,Q138,Q260,Q262
0,2.0,4.0,2.0,37.0
1,2.0,4.0,2.0,42.0
2,2.0,3.0,1.0,28.0
3,2.0,3.0,2.0,30.0
4,1.0,4.0,2.0,32.0


In [5]:
for var in variables_to_check:
    print("\n--------------------")
    print(var)
    print(sg[var].value_counts(dropna=False).sort_index())


--------------------
Q131
Q131
-2.0       2
 1.0     756
 2.0    1093
 3.0     152
 4.0       9
Name: count, dtype: int64

--------------------
Q138
Q138
-2.0      11
 1.0      16
 2.0      66
 3.0     466
 4.0    1453
Name: count, dtype: int64

--------------------
Q260
Q260
1.0     924
2.0    1088
Name: count, dtype: int64

--------------------
Q262
Q262
21.0    36
22.0    42
23.0    42
24.0    40
25.0    39
        ..
85.0     2
87.0     2
89.0     2
90.0     1
91.0     1
Name: count, Length: 69, dtype: int64


In [23]:
clean = sg[["D_INTERVIEW", "Q131", "Q138", "Q260", "Q262"]].copy()

clean.head()

,D_INTERVIEW,Q131,Q138,Q260,Q262
0,702070001.0,2.0,4.0,2.0,37.0
1,702070002.0,2.0,4.0,2.0,42.0
2,702070003.0,2.0,3.0,1.0,28.0
3,702070004.0,2.0,3.0,2.0,30.0
4,702070005.0,1.0,4.0,2.0,32.0


In [7]:
# Check original values
for var in ["Q131", "Q138", "Q260", "Q262"]:
    print("\n", var)
    print(clean[var].value_counts(dropna=False).sort_index())


 Q131
Q131
-2.0       2
 1.0     756
 2.0    1093
 3.0     152
 4.0       9
Name: count, dtype: int64

 Q138
Q138
-2.0      11
 1.0      16
 2.0      66
 3.0     466
 4.0    1453
Name: count, dtype: int64

 Q260
Q260
1.0     924
2.0    1088
Name: count, dtype: int64

 Q262
Q262
21.0    36
22.0    42
23.0    42
24.0    40
25.0    39
        ..
85.0     2
87.0     2
89.0     2
90.0     1
91.0     1
Name: count, Length: 69, dtype: int64


In [8]:
# Recode negative values as missing
clean[["Q131", "Q138", "Q260", "Q262"]] = (
    clean[["Q131", "Q138", "Q260", "Q262"]]
    .mask(clean[["Q131", "Q138", "Q260", "Q262"]] < 0)
)

In [9]:
for var in ["Q131", "Q138", "Q260", "Q262"]:
    print("\n", var)
    print(clean[var].value_counts(dropna=False).sort_index())


 Q131
Q131
1.0     756
2.0    1093
3.0     152
4.0       9
NaN       2
Name: count, dtype: int64

 Q138
Q138
1.0      16
2.0      66
3.0     466
4.0    1453
NaN      11
Name: count, dtype: int64

 Q260
Q260
1.0     924
2.0    1088
Name: count, dtype: int64

 Q262
Q262
21.0    36
22.0    42
23.0    42
24.0    40
25.0    39
        ..
85.0     2
87.0     2
89.0     2
90.0     1
91.0     1
Name: count, Length: 69, dtype: int64


In [10]:
# Keep only valid response ranges
clean.loc[~clean["Q131"].between(1, 4), "Q131"] = np.nan
clean.loc[~clean["Q138"].between(1, 4), "Q138"] = np.nan
clean.loc[~clean["Q260"].between(1, 2), "Q260"] = np.nan
clean.loc[~clean["Q262"].between(21, 100), "Q262"] = np.nan

In [11]:
clean = clean.rename(columns={
    "Q131": "subjective_security",
    "Q138": "sexual_harassment",
    "Q260": "gender",
    "Q262": "age"
})

clean.head()

,subjective_security,sexual_harassment,gender,age
0,2.0,4.0,2.0,37.0
1,2.0,4.0,2.0,42.0
2,2.0,3.0,1.0,28.0
3,2.0,3.0,2.0,30.0
4,1.0,4.0,2.0,32.0


In [13]:
clean["security_label"] = clean["subjective_security"].map({
    1: "Very secure",
    2: "Quite secure",
    3: "Not very secure",
    4: "Not at all secure"
})

clean["harassment_label"] = clean["sexual_harassment"].map({
    1: "Very frequently",
    2: "Quite frequently",
    3: "Not frequently",
    4: "Not at all frequently"
})

In [14]:
clean["gender_label"] = clean["gender"].map({
    1: "Male",
    2: "Female"
})

clean[["gender", "gender_label"]].head()

,gender,gender_label
0,2.0,Female
1,2.0,Female
2,1.0,Male
3,2.0,Female
4,2.0,Female


In [15]:
print(clean.columns.tolist())

['subjective_security', 'sexual_harassment', 'gender', 'age', 'security_cat', 'harassment_cat', 'gender_cat', 'security_label', 'harassment_label', 'gender_label']


In [16]:
for var in ["subjective_security", "sexual_harassment", "gender"]:
    print("\n--------------------")
    print(var)
    print(clean[var].value_counts(dropna=False).sort_index())


--------------------
subjective_security
subjective_security
1.0     756
2.0    1093
3.0     152
4.0       9
NaN       2
Name: count, dtype: int64

--------------------
sexual_harassment
sexual_harassment
1.0      16
2.0      66
3.0     466
4.0    1453
NaN      11
Name: count, dtype: int64

--------------------
gender
gender
1.0     924
2.0    1088
Name: count, dtype: int64


In [17]:
clean.isna().sum()

subjective_security     2
sexual_harassment      11
gender                  0
age                     0
security_cat            2
harassment_cat         11
gender_cat              0
security_label          2
harassment_label       11
gender_label            0
dtype: int64

In [18]:
clean.head(10)

,subjective_security,sexual_harassment,gender,age,security_cat,harassment_cat,gender_cat,security_label,harassment_label,gender_label
0,2.0,4.0,2.0,37.0,2,4,Female,Quite secure,Not at all frequently,Female
1,2.0,4.0,2.0,42.0,2,4,Female,Quite secure,Not at all frequently,Female
2,2.0,3.0,1.0,28.0,2,3,Male,Quite secure,Not frequently,Male
3,2.0,3.0,2.0,30.0,2,3,Female,Quite secure,Not frequently,Female
4,1.0,4.0,2.0,32.0,1,4,Female,Very secure,Not at all frequently,Female
5,1.0,4.0,2.0,53.0,1,4,Female,Very secure,Not at all frequently,Female
6,2.0,4.0,2.0,73.0,2,4,Female,Quite secure,Not at all frequently,Female
7,1.0,4.0,1.0,41.0,1,4,Male,Very secure,Not at all frequently,Male
8,2.0,3.0,1.0,31.0,2,3,Male,Quite secure,Not frequently,Male
9,2.0,4.0,2.0,57.0,2,4,Female,Quite secure,Not at all frequently,Female


In [19]:
clean.to_csv("../data/processed/clean_sg_part_a_yihan.csv", index=False)

In [20]:
pd.read_csv("../data/processed/clean_sg_part_a_yihan.csv").head()

,subjective_security,sexual_harassment,gender,age,security_cat,harassment_cat,gender_cat,security_label,harassment_label,gender_label
0,2.0,4.0,2.0,37.0,2.0,4.0,Female,Quite secure,Not at all frequently,Female
1,2.0,4.0,2.0,42.0,2.0,4.0,Female,Quite secure,Not at all frequently,Female
2,2.0,3.0,1.0,28.0,2.0,3.0,Male,Quite secure,Not frequently,Male
3,2.0,3.0,2.0,30.0,2.0,3.0,Female,Quite secure,Not frequently,Female
4,1.0,4.0,2.0,32.0,1.0,4.0,Female,Very secure,Not at all frequently,Female


In [21]:
sg.shape

(2012, 387)

In [22]:
[c for c in sg.columns if any(x in c.upper() for x in ["ID", "CASE", "INTERVIEW", "RESPONDENT"])]

['D_INTERVIEW']

In [24]:
clean.to_csv(
    "../data/processed/clean_sg_part_a_yihan.csv",
    index=False
)

In [25]:
pd.read_csv("../data/processed/clean_sg_part_a_yihan.csv").head()

,D_INTERVIEW,Q131,Q138,Q260,Q262
0,702070001.0,2.0,4.0,2.0,37.0
1,702070002.0,2.0,4.0,2.0,42.0
2,702070003.0,2.0,3.0,1.0,28.0
3,702070004.0,2.0,3.0,2.0,30.0
4,702070005.0,1.0,4.0,2.0,32.0


In [26]:
print("Rows:", len(clean))
print("Unique IDs:", clean["D_INTERVIEW"].nunique())
print("Duplicate IDs:", clean["D_INTERVIEW"].duplicated().sum())

Rows: 2012
Unique IDs: 2012
Duplicate IDs: 0
